# P1 · ROI Analysis — Post-Hackathon Resource Usage (Victoria Knockout)

Compares **KIDS26 legacy SNAP** (`input/…`) vs **Daedalus post-hackathon** (`../../analyses/sc-rna-seq-snap-daedalus-Victoria-Knockout/daedalus-post-hackathon/…`).

**ROI scopes:** Biohackathon = 2 modules (`upstream`, `integrative`); post-hackathon = 5 paired modules + FastQC/Cell Ranger (shared project data) + Daedalus biological validation. See `reports/P1-ROI-Biohackathon-vs-Post-Hackathon-Comparison.md`.

Plots → **`output/results-posthackathon/`** (`resource_utilization_comparison.png`, `memory_requested_vs_used.png`, `cost_improvement_pct.png`).

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import matplotlib.pyplot as plt

from scripts.load_resource_usage_csv import (
    COMPARE_MODULES,
    load_daedalus_csv,
    load_legacy_csv,
)

INPUT_DIR = ROOT_DIR / "input"
OUTPUT_DIR = ROOT_DIR / "output" / "results-posthackathon"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEAM15_ROOT = ROOT_DIR.parent.parent  # roi-analysis → deliverables → KIDS26-Team15
LEGACY_CSV = INPUT_DIR / "resource_usage_KIDS26-Victoria-Knockout_2026-11-05.csv"
DAEDALUS_CSV = (
    TEAM15_ROOT
    / "analyses"
    / "sc-rna-seq-snap-daedalus-Victoria-Knockout"
    / "daedalus-post-hackathon"
    / "resource_usage_2026-11-05_220340654817588.csv"
)

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
print("Output:", OUTPUT_DIR)

In [ ]:
legacy = load_legacy_csv(LEGACY_CSV)
daedalus = load_daedalus_csv(DAEDALUS_CSV)
modules = [m for m in COMPARE_MODULES if m in legacy or m in daedalus]

rows = []
for mod in modules:
    for label, data in (("Legacy", legacy), ("Daedalus", daedalus)):
        if mod not in data:
            continue
        m = data[mod]
        rows.append({
            "Module": mod,
            "Pipeline": label,
            "Cores": m["cores"],
            "Mem slot (GB)": m["mem_slot_gb"],
            "Wall (h)": round(m["wall_h"], 2),
            "Avg mem (GB)": m["avg_mem_gb"],
            "Total req mem (GB)": m["total_req_mem_gb"],
            "Memory util (%)": round(m["memory_util_pct"], 2),
            "CPU util (%)": round(m["cpu_util_pct"], 2),
            "Core-hours": round(m["core_hours"], 2),
        })

df_compare = pd.DataFrame(rows)
df_compare

In [ ]:
# Regenerate all charts (same logic as scripts/generate_posthackathon_plots.py)
from scripts import generate_posthackathon_plots as gp

gp.main()
list(OUTPUT_DIR.glob("*.png"))

## Notes

- Legacy memory util uses **avg memory / (cores × per-slot GB)** for parity with the original ROI notebook.
- Legacy-only modules (`fastqc`, `cellranger`) are omitted from module bar charts but included in legacy **total core-hours** for HPC cost.
- See `reports/P1-ROI-Biohackathon-vs-Post-Hackathon-Comparison.md` for the full table.